In [1]:
import SynGenLoss as sgl
from SynGenLoss.SatModels import SaturationModel2, SatModelDataClass2
from SynGenLoss.GenModel2 import GenDataClass2, GeneratorModel2, GeneratorModel2_If_in

import numpy as np 


In [2]:
i_0 = 1.0
i_10 = 1.2
i_12 = 1.8

SG10 = i_10/i_0 - 1
SG12 = i_12/(1.2 * i_0) - 1

# 103 MVA Generator: 
model_data = GenDataClass2() 
model_data.standard_params(S_n_mva=103, V_nom_kV=11, cos_phi=0.9, I_f_base=525.15, R_a_nom=0.00182, R_f_nom=None, X_d_u=1.087, 
                           X_q_u=0.676, X_l=0.08)
model_data.nominal_losses(V_nom=1.0, I_fd_pu_nom=1065/525.15, P_sn_kW=276.62, P_rn_kW=175.78, P_exn_kW=15.88, P_cn_kW=211.92, P_const_kW=413.82)
# model_data.nominal_losses(V_nom=1.0, I_fd_pu_nom=1065/525.15, P_sn_kW=276.62, P_rn_kW=175.78, P_exn_kW=15.88, P_cn_kW=211.92, P_const_kW=172.92+356.23)

sat_model_data = SatModelDataClass2(X_d_u=1.059, X_q_u=0.676, X_l=0.08, R_a=0.00182, SG10=SG10, SG12=SG12)
sat_model = SaturationModel2(sat_model_data)

Gen103MVA = GeneratorModel2(model_data, sat_model)
Gen103MVA_If_in = GeneratorModel2_If_in(model_data) 

In [3]:
P = 0.9 
Q = P * np.tan(np.arccos(0.9))
I_a, I_f, delta = Gen103MVA._calc_currents(P, Q, 1.0)
P_losses = Gen103MVA.get_P_losses(P, Q, 1.0)

print(P_losses*103000)

1053.4464574588792
P_in = 92700.000, P_loss_tot = 1090.055, eff = 98.838 %
P_l_stator = 276.620, P_l_rotor = 171.987, P_l_ex = 15.708, P_l_core = 211.920, P_const = 413.820


In [4]:
Gen103MVA._calc_currents(P, Q, 1.0)

(1.0, 2.0059915404339317, 0.43831521898338777)

In [5]:
P = 0.9 
Q = P * np.tan(np.arccos(0.9))
P_losses = Gen103MVA_If_in.get_P_losses(P, Q, 1.0, )
print(P_losses*103000)

TypeError: GeneratorModel2_If_in.get_P_losses() missing 1 required positional argument: 'I_f'

In [6]:
P_loss = Gen103MVA_If_in.get_P_losses(P, Q, 1.0, 2.02786362106938)
print(P_loss*103000) 

P_in = 92700.000, P_loss_tot = 1093.997, eff = 98.834 %
P_l_stator = 276.620, P_l_rotor = 175.758, P_l_ex = 15.879, P_l_core = 211.920, P_const = 413.820


In [7]:
I_f_tests = [1065.0, 936.12, 816.18, 711.38, 873.17, 776.61, 698.7, 646.84]
I_f_tests = [1055.0, 936.12, 816.18, 711.38, 873.17, 776.61, 698.7, 646.84]
I_f_calc_tests = [1064.88, 934.23, 815.01, 711.11, 857.50, 764.76, 691.88, 644.67]
P_tests = [0.900, 0.675, 0.450, 0.225, 1.000, 0.750, 0.500, 0.250]
cos_phi_tests = [0.9, 0.9, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0]
Q_tests = [np.tan(np.arccos(cos_phi))*P for cos_phi, P in zip(cos_phi_tests, P_tests)]  
I_a_tests = [5406.1, 4054.6, 2703.0, 1351.5, 5406.1, 4054.6, 2703.0, 1351.5]
phi_tests = [np.arctan(Q/P) for Q, P in zip(Q_tests, P_tests)]
I_a_tests_pu = [np.sqrt(P**2 + Q**2)/V for P, Q, V in zip(P_tests, Q_tests, [1.0]*8)]

In [8]:
def make_model(SG10, SG12, X_d_u): 
    sat_model_data = SatModelDataClass2(X_d_u=X_d_u, X_q_u=0.676, X_l=0.08, R_a=0.00182, SG10=SG10, SG12=SG12)
    sat_model = SaturationModel2(sat_model_data)
    return sat_model

def squared_error(X): 
    SG10, SG12, X_d_u = X
    SG10 = np.clip(SG10, 0.001, 10)
    SG12 = np.clip(SG12, 0.001, 10)
    X_d_u = np.clip(X_d_u, 0.8, 1.2)
    sat_model = make_model(SG10, SG12, X_d_u)
    I_fd_vals = []
    I_f_tests = np.array([1065.0, 936.12, 816.18, 711.38, 873.17, 776.61, 698.7, 646.84])
    for I_a, phi, I_f_calc, I_f_meas in zip(I_a_tests_pu, phi_tests, I_f_calc_tests, I_f_tests): 
        I_fd, delta, psi_m = sat_model.calc_ifd(1.0, I_a, phi)
        # print(f"This calc I_f: {I_fd*model_data.I_f_base:.3f} A, Yannick calc: {I_f_calc}, meas = {I_f_meas}") 
        I_fd_vals.append(I_fd*525.15) 
    I_fd_vals = np.array(I_fd_vals) 
    return np.sum((I_fd_vals - I_f_tests)**2)


In [9]:
from scipy.optimize import minimize, root

In [10]:
X_d_0 = 1.059
bnds = [(SG10*0.5, SG10*1.5), (SG12*0.5, SG12*1.5), (X_d_0*0.5, X_d_0*1.5)]

sol = minimize(squared_error, np.array([SG10, SG12, X_d_0]), bounds=bnds, tol=1e-9) 
sol 

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 548.3078902545293
        x: [ 1.662e-01  7.500e-01  1.037e+00]
      nit: 16
      jac: [ 1.287e-02 -1.298e+02 -1.501e-02]
     nfev: 92
     njev: 23
 hess_inv: <3x3 LbfgsInvHessProduct with dtype=float64>

In [98]:
sol.x

array([0.16618528, 0.75      , 1.03725934])

In [99]:
bnds 

[(0.09999999999999998, 0.29999999999999993),
 (0.25, 0.75),
 (0.5295, 1.5884999999999998)]

In [102]:
root(lambda x: x**2, 4)

 message: The number of calls to function has reached maxfev = 400.
 success: False
  status: 2
     fun: [ 5.164e-166]
       x: [ 2.272e-83]
    nfev: 400
    fjac: [[-1.000e+00]]
       r: [-9.626e-83]
     qtf: [-1.352e-165]